# ArmBench — ML Inference Profiling

This notebook runs ArmBench on a GPU so you can see real CPU vs GPU performance comparisons.

**Runtime → Change runtime type → GPU** before running.

In [ ]:
!git clone https://github.com/rohan9446/armbench.git
%cd armbench
!pip install -e . -q

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## CPU Backends

Baseline profiling on CPU — FP32, INT8, ONNX Runtime, and pruned.

In [ ]:
!armbench run --model resnet18 --backends fp32 int8 onnx_fp32 pruned --runs 30

## GPU Backends

This is where FP16 shines — GPUs have dedicated FP16 compute units.

In [ ]:
!armbench run --model resnet18 --backends cuda_fp32 cuda_fp16 --runs 30

## Full Comparison — All Backends

In [ ]:
!armbench run --model resnet50 --backends fp32 int8 onnx_fp32 pruned cuda_fp32 cuda_fp16 --runs 30

## View Results

In [ ]:
import json
from IPython.display import HTML, display

# show the JSON report
with open("results/resnet50_report.json") as f:
    data = json.load(f)

for b in data["benchmarks"]:
    lat = b["latency"]
    print(f"{b['backend']:12s}  p50={lat['p50_ms']:8.2f}ms  p95={lat['p95_ms']:8.2f}ms  "
          f"mem={b['peak_memory_mb']:7.1f}MB  throughput={b['throughput_ops_sec']:7.1f} ops/s  "
          f"disk={b.get('disk_size_mb', '-')}MB")

In [ ]:
# render the HTML dashboard inline
with open("results/resnet50_report.html") as f:
    display(HTML(f.read()))